In [5]:
import pandas as pd
import numpy as np

df = pd.read_excel('Online Retail.xlsx')
print(df.shape)
display(df.head())
df.info()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [6]:
print("Sebelum:", df.shape, df['InvoiceNo'].nunique(), df['CustomerID'].nunique())
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
print("Sesudah:", df.shape, df['InvoiceNo'].nunique(), df['CustomerID'].nunique())

Sebelum: (541909, 8) 25900 4372
Sesudah: (397924, 8) 18536 4339


In [7]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df_sorted = df.sort_values(['CustomerID', 'InvoiceDate', 'InvoiceNo'])
sequences = (
    df_sorted.groupby(['CustomerID', 'InvoiceDate', 'InvoiceNo'])['Description']
    .apply(list)
    .groupby('CustomerID')
    .apply(lambda x: list(x))
    .tolist()
)
print("Total sequences (customer):", len(sequences))
print("Contoh sequence customer pertama:", sequences[0][:2])

Total sequences (customer): 4339
Contoh sequence customer pertama: [['MEDIUM CERAMIC TOP STORAGE JAR']]


In [ ]:
!pip install prefixspan -q
from prefixspan import PrefixSpan

df = df.dropna(subset=['CustomerID', 'Description'])
df['Description'] = df['Description'].astype(str).str.strip()
sequences = (
    df_sorted.groupby(['CustomerID', 'InvoiceDate', 'InvoiceNo'])['Description']
    .apply(list)
    .groupby('CustomerID')
    .apply(lambda x: [item for sublist in x for item in sublist if isinstance(item, str) and item])
    .tolist()
)
sequences = [seq for seq in sequences if len(seq) > 0]
print("Total sequences bersih:", len(sequences))
ps = PrefixSpan(sequences)
minsup = int(len(sequences) * 0.01)
print("Minsup threshold:", minsup)
patterns = ps.frequent(minsup)
patterns.sort(key=lambda x: x[0], reverse=True)
for sup, pat in patterns[:15]:
    print(f"Support: {sup} | Pattern: {pat}")


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Total sequences bersih: 4339
Minsup threshold: 43
